# Hindi Claim Verification

### Architecture
```
Wikipedia Infoboxes
       ↓
  Knowledge Base  (Stage 1)
       ↓                    ↓
HiNER entity typing    IndoWordNet lookup
(for entity pool)      (synonyms, hypernyms,
                        hyponyms, siblings, similar)
       ↓                    ↓
  Claims DB  ←────── Transformations (Stage 5)
  (Stage 6)                 ↑
       ↓              Numerical Mutation
  Evidence DB            Entity Swap (HiNER)
  (Stage 3)              Relation Mutation (IWN hypernymy/siblings)
       ↓                 Semantic Contrast (IWN co-hyponyms/similar)
  Final Dataset          Paraphrase(IWN synonyms) + Corruption
  (Stage 7 — JOIN)
       ↓
  NLI Training Files
```

### IndoWordNet Relations Used
| IWN Relation | How We Use It |
|---|---|
| `synsets()` → `lemma_names()` | Synonyms of a key word — vary evidence surface form |
| `SynsetRelations.HYPERNYMY` | Infer semantic category of a key (पत्नी → परिवार-सदस्य) |
| `SynsetRelations.HYPONYMY` | Get sub-types; combine with HYPERNYMY to find siblings |
| `SynsetRelations.SIMILAR` | Semantically related but distinct alternatives |
| Co-hyponyms (siblings) | Best semantic contrast — same category, different concept |

### HiNER Usage
| HiNER Output | How We Use It |
|---|---|
| B-PER / I-PER | Tag value as PERSON → route to person entity pool |
| B-LOC / I-LOC | Tag value as LOCATION → route to location pool |
| B-ORG / I-ORG | Tag value as ORGANIZATION → route to org pool |
| Type of distractor | Verify replacement has same NER type as original |
| Type of key context | Infer what entity type the infobox key expects |

### Key Design Rules
- HiNER types all entities — no manual type lists
- IWN siblings replace hardcoded contrasts — no manual antonym maps
- IWN synonyms vary key surface form in evidence — different from claim template
- IWN HYPERNYMY determines key semantic roles — no manual key groups
- Five-point validation on every FALSE claim
- Page-level stratified split

---
##  Install

In [ ]:
!pip install wikipedia-api beautifulsoup4 requests tqdm pandas numpy -q
!pip install transformers torch datasets -q
!pip install pyiwn -q          # IndoWordNet Python wrapper
!pip install indic-nlp-library -q
!pip install scikit-learn matplotlib seaborn -q
print('Done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 15.4 MB/s eta 0:00:00
Done


## Imports and Tool Initialization

In [ ]:
import wikipediaapi, requests, json, re, random, time, warnings
from pathlib import Path
from collections import defaultdict
from typing import Optional, List, Dict, Tuple, Set

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# ── IndoWordNet ───────────────────────────────────────────────────────────────
IWN = None
IWN_AVAILABLE = False
try:
    import pyiwn
    # pyiwn downloads ~31MB data on first run — allow a minute
    IWN = pyiwn.IndoWordNet()          # loads Hindi synsets by default
    IWN_AVAILABLE = True
    print('IndoWordNet loaded')
    # Show all available relation types — these are what we work with
    all_relations = [r.value for r in pyiwn.SynsetRelations]
    print(f'Relations available: {all_relations}')
except Exception as e:
    print(f'IndoWordNet unavailable: {e}')
    print('Ensure network access for Dropbox download on first run')

# ── HiNER ────────────────────────────────────────────────────────────────────
NER_PIPELINE = None
HINER_AVAILABLE = False
try:
    from transformers import pipeline
    NER_PIPELINE = pipeline(
        'ner',
        model='cfilt/HiNER-original-xlm-roberta-large',
        aggregation_strategy='simple'
    )
    HINER_AVAILABLE = True
    print('HiNER loaded')
except Exception as e:
    print(f'HiNER unavailable: {e}')

# ── Indic NLP normalizer ──────────────────────────────────────────────────────
NORMALIZER = None
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    NORMALIZER = IndicNormalizerFactory().get_normalizer('hi')
    print('Indic NLP normalizer loaded')
except Exception as e:
    print(f'Indic NLP unavailable: {e}')

# ── Wikipedia ────────────────────────────────────────────────────────────────
WIKI = wikipediaapi.Wikipedia(language='hi', user_agent='HindiCV/4.0 (research)')

OUT_DIR = Path('hindi_cv_v4')
OUT_DIR.mkdir(exist_ok=True)
print(f'Output: {OUT_DIR}')

[██████████████████████████████████████████████████]
✓ IndoWordNet loaded
  Relations available: ['mero_member_collection', 'ability_verb', 'causative', 'capability_verb', 'mero_component_object', 'holo_portion_mass', 'function_verb', 'holo_component_object', 'hypernymy', 'entailment', 'also_see', 'mero_feature_activity', 'holo_place_area', 'modifies_verb', 'attributes', 'mero_portion_mass', 'modifies_noun', 'holo_feature_activity', 'mero_stuff_object', 'troponymy', 'mero_place_area', 'holo_member_collection', 'hyponymy', 'similar', 'mero_position_area', 'holo_position_area', 'holo_stuff_object']


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: cfilt/HiNER-original-xlm-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✓ HiNER loaded
✓ Indic NLP normalizer loaded
✓ Output: hindi_cv_v4


## Configuration

In [ ]:
PAGES_PER_TOPIC       = 40
MAX_FACTS_PER_PAGE    = 8
MIN_FACTS_FOR_PASSAGE = 3
FALSE_PER_TRUE        = 2
REQUEST_DELAY         = 1.0
MAX_VALUE_LEN         = 100
MIN_VALUE_LEN         = 3

CATEGORIES = {
    'entertainment': 'श्रेणी:भारतीय फ़िल्म अभिनेता',
    'sports':        'श्रेणी:भारतीय क्रिकेट खिलाड़ी',
    'politics':      'श्रेणी:भारतीय राजनीतिज्ञ',
    'science':       'श्रेणी:भारतीय वैज्ञानिक',
    'literature':    'श्रेणी:हिन्दी साहित्यकार',
    'geography':     'श्रेणी:भारत के राज्य',
}

IGNORE_FIELDS = {
    'चित्र','image','caption','size','alt','चित्र आकार',
    'हस्ताक्षर','signature','website','वेबसाइट',
    'प्रतियोगिता','मैच','रन','औसत','गेंद','विकेट',
    'कैच','शतक','उच्च स्कोर','पारी','नाबाद'
}

HINDI_MONTHS = [
    'जनवरी','फ़रवरी','फरवरी','मार्च','अप्रैल','मई','जून',
    'जुलाई','अगस्त','सितम्बर','सितंबर','अक्टूबर','अक्तूबर',
    'नवम्बर','नवंबर','दिसम्बर','दिसंबर'
]

_DEVA_RE = re.compile(r'[\u0900-\u097F]')
_NUM_RE  = re.compile(r'[0-9०-९]')
_IMG_RE  = re.compile(r'\.(jpg|jpeg|png|svg|gif|webp)$', re.I)
_JUNK_RE = re.compile(r'^(?:yes|no|true|false|left|right|center|auto|none|[a-z]{1,4}px)$', re.I)
_PURE_EN = re.compile(r'^[a-zA-Z0-9\s\-\.]+$')

DEVA_DIGITS  = '०१२३४५६७८९'
LATIN_DIGITS = '0123456789'
D2L = str.maketrans(DEVA_DIGITS, LATIN_DIGITS)
L2D = str.maketrans(LATIN_DIGITS, DEVA_DIGITS)

print('Configuration ready')

Configuration ready


---
# PART A — IndoWordNet Cache

## What IWN provides that we use:

**`synsets(word)` → list of Synset objects**
Each Synset has:
- `.lemma_names()` → all synonyms in that synset
  - Used for: key surface variation in evidence (पत्नी → धर्मपत्नी, जीवनसाथी, बीवी ...)
- `.synset_id()` → integer ID for relation lookup

**`synset_relation(synset, SynsetRelations.HYPERNYMY)`**
- Broader concept synsets
- Used for: infer what entity type a key expects
  (if पत्नी's hypernym chain includes 'व्यक्ति' → key expects a PERSON)

**`synset_relation(synset, SynsetRelations.HYPONYMY)`**
- Narrower concept synsets
- Used for: finding siblings (co-hyponyms) = best semantic contrast
  (siblings of पत्नी under family-member = माता, पुत्री, बहन ...)

**`synset_relation(synset, SynsetRelations.SIMILAR)`**
- Semantically related words in same POS
- Used for: fallback semantic contrast

**Note on antonyms:** pyiwn's `Lemma.antonym()` raises `NotImplementedError`.
We use IWN co-hyponyms (siblings) instead — semantically equivalent for our purpose.

## IndoWordNet Cache Engine

In [ ]:
class IWNCache:
    """
    Wraps pyiwn with caching and a clean interface.

    For each Hindi word queried, stores:
      synonyms   — all lemma names in the same synset
      hypernyms  — lemma names of all hypernym synsets
      hyponyms   — lemma names of all hyponym synsets
      similar    — lemma names of all similar synsets
      siblings   — co-hyponyms: other hyponyms of the same hypernym
                   This is the key resource for semantic contrast.
                   Example: पत्नी → siblings under family-member
                            = माता, बहन, पुत्री, etc.
    """

    def __init__(self, iwn):
        self._iwn = iwn
        self._cache: Dict[str, Dict] = {}

    def _fetch(self, word: str) -> Dict:
        if word in self._cache:
            return self._cache[word]

        empty = {'synonyms':[],'hypernyms':[],'hyponyms':[],'similar':[],'siblings':[],'found':False}

        if self._iwn is None:
            self._cache[word] = empty
            return empty

        try:
            synsets = self._iwn.synsets(word)
        except (KeyError, Exception):
            self._cache[word] = empty
            return empty

        if not synsets:
            self._cache[word] = empty
            return empty

        result = {'synonyms':[],'hypernyms':[],'hyponyms':[],'similar':[],'siblings':[],'found':True}

        for synset in synsets:
            # ── Synonyms (all lemma names in synset) ──────────────────────
            result['synonyms'].extend(
                [l for l in synset.lemma_names() if l != word]
            )

            # ── Hypernyms ─────────────────────────────────────────────────
            hyper_ss = self._iwn.synset_relation(synset, pyiwn.SynsetRelations.HYPERNYMY)
            for hs in hyper_ss:
                result['hypernyms'].extend(hs.lemma_names())

                # ── Siblings = co-hyponyms of each hypernym ───────────────
                # These are the best semantic contrasts:
                # words in the same category, mutually exclusive.
                sibling_ss = self._iwn.synset_relation(hs, pyiwn.SynsetRelations.HYPONYMY)
                for ss in sibling_ss:
                    for lemma in ss.lemma_names():
                        if lemma != word:
                            result['siblings'].append(lemma)

            # ── Hyponyms ──────────────────────────────────────────────────
            hypo_ss = self._iwn.synset_relation(synset, pyiwn.SynsetRelations.HYPONYMY)
            for hs in hypo_ss:
                result['hyponyms'].extend(hs.lemma_names())

            # ── Similar ───────────────────────────────────────────────────
            sim_ss = self._iwn.synset_relation(synset, pyiwn.SynsetRelations.SIMILAR)
            for ss in sim_ss:
                result['similar'].extend([l for l in ss.lemma_names() if l != word])

        # Deduplicate
        for k in result:
            if isinstance(result[k], list):
                result[k] = list(dict.fromkeys(result[k]))

        self._cache[word] = result
        return result

    def synonyms(self, word: str) -> List[str]:
        """All synonyms from IWN synset. Used for key surface variation."""
        return self._fetch(word)['synonyms']

    def hypernyms(self, word: str) -> List[str]:
        """Broader concepts. Used to infer entity type expected by a key."""
        return self._fetch(word)['hypernyms']

    def hyponyms(self, word: str) -> List[str]:
        """More specific concepts under this word."""
        return self._fetch(word)['hyponyms']

    def siblings(self, word: str) -> List[str]:
        """
        Co-hyponyms: words that share the same hypernym as this word.
        These are the primary source of semantic contrast in this pipeline.
        Example: पत्नी → [माता, बहन, पुत्री, ...] (all are family members)
        """
        return self._fetch(word)['siblings']

    def similar(self, word: str) -> List[str]:
        """Semantically related words. Fallback for semantic contrast."""
        return self._fetch(word)['similar']

    def found(self, word: str) -> bool:
        """Check if word exists in IWN."""
        return self._fetch(word)['found']

    def get_semantic_contrast(self, word: str) -> Optional[str]:
        """
        Best semantic contrast using IWN — zero hardcoding.

        Priority:
        1. IWN siblings (co-hyponyms) — same category, different concept.
           Example: 'बल्लेबाज' siblings = 'गेंदबाज', 'विकेटकीपर'
           These are mutually exclusive and maximally plausible.

        2. IWN similar — related but distinct.

        3. IWN hyponyms — pick a different sub-type.

        Returns a contrast word (Devanagari) or None.
        """
        # 1. Siblings
        sibs = [s for s in self.siblings(word) if _DEVA_RE.search(s) and s != word]
        if sibs:
            return random.choice(sibs[:10])

        # 2. Similar
        sims = [s for s in self.similar(word) if _DEVA_RE.search(s) and s != word]
        if sims:
            return random.choice(sims[:10])

        # 3. Hyponyms
        hypos = [h for h in self.hyponyms(word) if _DEVA_RE.search(h) and h != word]
        if hypos:
            return random.choice(hypos[:10])

        return None

    def get_key_synonym(self, key_word: str) -> Optional[str]:
        """
        Get a synonym of a key word for surface variation in evidence.
        Example: 'पत्नी' → 'धर्मपत्नी' or 'जीवनसाथी' or 'बीवी'
        This makes the evidence sentence key different from the claim sentence key.
        """
        syns = [s for s in self.synonyms(key_word) if _DEVA_RE.search(s) and len(s) <= 20]
        return random.choice(syns) if syns else None

    def infer_expected_entity_type(self, key_word: str) -> str:
        """
        Infer what entity type an infobox key expects,
        using IWN hypernym path of the key word.

        We climb the hypernym tree and check if any ancestor
        falls in the PERSON, LOCATION, or ORGANIZATION category.

        Returns 'PERSON' | 'LOCATION' | 'ORGANIZATION' | 'UNKNOWN'
        """
        PERSON_ANCHORS  = {'व्यक्ति','मानव','नारी','पुरुष','स्त्री','जन'}
        LOCATION_ANCHORS = {'स्थान','देश','शहर','नगर','राज्य','क्षेत्र','भूमि'}
        ORG_ANCHORS     = {'संगठन','संस्था','दल','पार्टी','प्रतिष्ठान'}

        hypers = self.hypernyms(key_word)
        for h in hypers:
            if h in PERSON_ANCHORS:   return 'PERSON'
            if h in LOCATION_ANCHORS: return 'LOCATION'
            if h in ORG_ANCHORS:      return 'ORGANIZATION'

        # Also check synonyms' hypernyms (broadens coverage)
        for syn in self.synonyms(key_word)[:5]:
            for h in self.hypernyms(syn):
                if h in PERSON_ANCHORS:   return 'PERSON'
                if h in LOCATION_ANCHORS: return 'LOCATION'
                if h in ORG_ANCHORS:      return 'ORGANIZATION'

        return 'UNKNOWN'

    def preload(self, words: List[str]) -> int:
        hits = 0
        for w in tqdm(words, desc='Preloading IWN cache'):
            if self._fetch(w)['found']:
                hits += 1
        print(f'  IWN coverage: {hits}/{len(words)} ({100*hits/max(1,len(words)):.1f}%) words found')
        return hits

    def stats(self) -> Dict:
        total = len(self._cache)
        found = sum(1 for v in self._cache.values() if v.get('found'))
        return {'queried': total, 'found': found, 'coverage': round(found/max(1,total), 3)}


IWN_CACHE = IWNCache(IWN)

# Test
if IWN_AVAILABLE:
    print('\nIndoWordNet Cache Test:')
    for word in ['पत्नी', 'माता', 'बल्लेबाज', 'राजधानी']:
        syns     = IWN_CACHE.synonyms(word)
        siblings = IWN_CACHE.siblings(word)
        contrast = IWN_CACHE.get_semantic_contrast(word)
        key_syn  = IWN_CACHE.get_key_synonym(word)
        inferred = IWN_CACHE.infer_expected_entity_type(word)
        print(f'  {word}')
        print(f'    synonyms (for evidence variation): {syns[:4]}')
        print(f'    siblings (for semantic contrast) : {siblings[:4]}')
        print(f'    best contrast                   : {contrast}')
        print(f'    inferred entity type of key     : {inferred}')


IndoWordNet Cache Test:
  पत्नी
    synonyms (for evidence variation): ['बीवी', 'बीबी', 'जोरू', 'घरवाली']
    siblings (for semantic contrast) : []
    best contrast                   : लुहारिन
    inferred entity type of key     : UNKNOWN
  माता
    synonyms (for evidence variation): ['शीतला', 'चेचक माई', 'शीतला देवी', 'शीतला माता']
    siblings (for semantic contrast) : []
    best contrast                   : वीरकुक्षि
    inferred entity type of key     : UNKNOWN
  बल्लेबाज
    synonyms (for evidence variation): ['बल्लेबाज़', 'बैट्समैन']
    siblings (for semantic contrast) : []
    best contrast                   : सलामी बल्लेबाज़
    inferred entity type of key     : UNKNOWN
  राजधानी
    synonyms (for evidence variation): ['महापुरी']
    siblings (for semantic contrast) : []
    best contrast                   : जॉर्ज टॉउन
    inferred entity type of key     : UNKNOWN


---
# PART B — HiNER Typing Engine

## HiNER Wrapper

In [ ]:
class HiNERTyper:
    """
    Uses HiNER for entity type classification.

    Three uses in this pipeline:

    1. type_value(value) → NER type of an infobox value
       Used to classify extracted facts for the entity pool.

    2. type_key(key) → expected NER type for an infobox key
       Combined with IWN hypernyms for maximum coverage:
       First checks IWN hypernym path, then falls back to HiNER on key text.

    3. same_type(v1, v2) → bool
       Verify that a distractor entity has the same NER type as original.
       Used in entity swap cross-verification.

    Label mapping:
       *PER* → PERSON
       *LOC* → LOCATION
       *ORG* → ORGANIZATION
       else  → OTHER
    """

    def __init__(self, pipeline):
        self._pipe  = pipeline
        self._cache: Dict[str, str] = {}

    def _run(self, text: str) -> str:
        if text in self._cache:
            return self._cache[text]

        if self._pipe is None:
            return 'OTHER'

        try:
            results = self._pipe(text)
            if not results:
                self._cache[text] = 'OTHER'
                return 'OTHER'

            type_scores = defaultdict(float)
            for r in results:
                eg = r.get('entity_group', '').upper()
                score = r.get('score', 0)
                if 'PER' in eg: type_scores['PERSON']       += score
                elif 'LOC' in eg: type_scores['LOCATION']   += score
                elif 'ORG' in eg: type_scores['ORGANIZATION'] += score

            ner_type = max(type_scores, key=type_scores.get) if type_scores else 'OTHER'
        except Exception:
            ner_type = 'OTHER'

        self._cache[text] = ner_type
        return ner_type

    def type_value(self, value: str) -> str:
        """NER type of an infobox value."""
        return self._run(value)

    def type_key(self, key: str) -> str:
        """
        Infer expected entity type for an infobox key.

        Strategy:
        1. Ask IWN for the hypernym path of the key word.
           IWN knows that 'पत्नी' → 'स्त्री' → 'व्यक्ति' → PERSON
           IWN knows that 'राजधानी' → 'शहर' → 'स्थान' → LOCATION
           This is zero hardcoding — the knowledge comes from IWN.

        2. Fallback to HiNER on the key text itself.
        """
        # First: IWN hypernym inference (more reliable for nominal keys)
        for kw in key.split():
            if not _DEVA_RE.search(kw):
                continue
            inferred = IWN_CACHE.infer_expected_entity_type(kw)
            if inferred != 'UNKNOWN':
                return inferred

        # Fallback: HiNER on key text
        return self._run(key)

    def same_type(self, v1: str, v2: str) -> bool:
        t1, t2 = self.type_value(v1), self.type_value(v2)
        # Both OTHER is ambiguous — don't reject
        if t1 == 'OTHER' or t2 == 'OTHER':
            return True
        return t1 == t2


TYPER = HiNERTyper(NER_PIPELINE)

if HINER_AVAILABLE:
    print('HiNER typing test:')
    for val in ['जया बच्चन','तिरुवनंतपुरम','भारतीय जनता पार्टी','बल्लेबाज','11 अक्टूबर 1942']:
        print(f'  type_value({val}) → {TYPER.type_value(val)}')
    print('\nKey type inference (IWN + HiNER):')
    for key in ['पत्नी','माता','राजधानी','जन्म स्थान','राजनीतिक दल']:
        print(f'  type_key({key}) → {TYPER.type_key(key)}')

HiNER typing test:
  type_value(जया बच्चन) → PERSON
  type_value(तिरुवनंतपुरम) → LOCATION
  type_value(भारतीय जनता पार्टी) → ORGANIZATION
  type_value(बल्लेबाज) → OTHER
  type_value(11 अक्टूबर 1942) → OTHER

Key type inference (IWN + HiNER):
  type_key(पत्नी) → OTHER
  type_key(माता) → OTHER
  type_key(राजधानी) → OTHER
  type_key(जन्म स्थान) → OTHER
  type_key(राजनीतिक दल) → OTHER


---
# PART C — Knowledge Base

## Value Cleaning and Classification

In [ ]:
def normalize(text: str) -> str:
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\(\d{4}-\d{2}-\d{2}\)', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if NORMALIZER:
        try: text = NORMALIZER.normalize(text)
        except: pass
    return text


def is_valid(v: str) -> bool:
    v = v.strip()
    if len(v) < MIN_VALUE_LEN or len(v) > MAX_VALUE_LEN: return False
    if _IMG_RE.search(v) or _JUNK_RE.match(v) or _PURE_EN.match(v): return False
    return bool(_DEVA_RE.search(v)) or bool(_NUM_RE.search(v))


def classify(key: str, value: str) -> Tuple[str, str]:
    """
    Classify (key, value) → (value_type, ner_type).

    value_type: date | numeric | person | place | organization | text
    ner_type:   PERSON | LOCATION | ORGANIZATION | OTHER

    Priority:
    1. Date: month name present, or year+date-key combination
    2. Numeric: pure digit string
    3. HiNER on the VALUE text → NER type
    4. IWN hypernym path of the KEY word → expected entity type
    5. Default: text / OTHER
    """
    v, k = value.strip(), key.strip()

    # 1. Date
    if any(m in v for m in HINDI_MONTHS): return 'date', 'OTHER'
    if re.search(r'\b(19|20)\d{2}\b', v) and any(dk in k for dk in ['जन्म','मृत्यु','स्थापना']):
        return 'date', 'OTHER'

    # 2. Numeric
    if re.match(r'^[0-9०-९][\d,०-९\.\-\s%]*$', v.strip()):
        return 'numeric', 'OTHER'

    # 3. HiNER on value
    ner_type = TYPER.type_value(v) if HINER_AVAILABLE else 'OTHER'
    if ner_type == 'PERSON':       return 'person',       ner_type
    if ner_type == 'LOCATION':     return 'place',        ner_type
    if ner_type == 'ORGANIZATION': return 'organization', ner_type

    # 4. IWN + HiNER on key
    key_type = TYPER.type_key(k)
    if key_type == 'PERSON':       return 'person',       key_type
    if key_type == 'LOCATION':     return 'place',        key_type
    if key_type == 'ORGANIZATION': return 'organization', key_type

    return 'text', 'OTHER'


print('Classification test:')
for k, v in [('जन्म','5 नवंबर 1988'),('भूमिका','बल्लेबाज'),
              ('जीवनसाथी','जया बच्चन'),('राजधानी','तिरुवनंतपुरम'),
              ('राजनीतिक दल','भारतीय जनता पार्टी'),('जनसंख्या','33,387,677')]:
    vt, nt = classify(k, v)
    print(f'  {k:15s}|{v:30s} → {vt:12s}/{nt}')

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Classification test:
  जन्म           |5 नवंबर 1988                   → date        /OTHER
  भूमिका         |बल्लेबाज                       → text        /OTHER
  जीवनसाथी       |जया बच्चन                      → person      /PERSON
  राजधानी        |तिरुवनंतपुरम                   → place       /LOCATION
  राजनीतिक दल    |भारतीय जनता पार्टी             → organization/ORGANIZATION
  जनसंख्या       |33,387,677                     → numeric     /OTHER


## Wikipedia Scraper

In [ ]:
def get_pages(cat: str, n: int) -> List[str]:
    pages = []
    for title, member in WIKI.page(cat).categorymembers.items():
        if member.ns == wikipediaapi.Namespace.MAIN:
            pages.append(member.title)
        if len(pages) >= n: break
    return pages


def scrape(title: str) -> Dict[str, str]:
    url = f'https://hi.wikipedia.org/wiki/{title.replace(" ","_")}'
    try:
        r = requests.get(url, headers={'User-Agent':'HindiCV/4.0'}, timeout=12)
        r.raise_for_status()
    except: return {}
    soup = BeautifulSoup(r.text, 'html.parser')
    ib   = soup.find('table', class_=lambda x: x and 'infobox' in x)
    if not ib: return {}
    data = {}
    for row in ib.find_all('tr'):
        th, td = row.find('th'), row.find('td')
        if not (th and td): continue
        key, val = normalize(th.get_text()), normalize(td.get_text())
        if any(ig in key for ig in IGNORE_FIELDS): continue
        if not is_valid(val): continue
        data[key] = val
    time.sleep(REQUEST_DELAY)
    return data


print('Scraper ready. Test:')
ib = scrape('अमिताभ बच्चन')
for k, v in list(ib.items())[:4]:
    vt, nt = classify(k, v)
    print(f'  {k:20s}: {v:30s} [{vt}/{nt}]')

Scraper ready. Test:
  जन्म                : 11 अक्टूबर 1942प्रयागराज       [date/OTHER]
  आवास                : मुम्बई                         [place/LOCATION]
  नागरिकता            : भारत                           [place/LOCATION]
  शिक्षा              : किरोड़ीमल कालेज                [organization/ORGANIZATION]


## Build Knowledge Base + IWN Preload

In [ ]:
def build_knowledge_base() -> pd.DataFrame:
    records = {}
    topic_pages = {}

    print('Phase 1 — Page lists')
    for topic, cat in CATEGORIES.items():
        try:
            p = get_pages(cat, PAGES_PER_TOPIC)
            topic_pages[topic] = p
            print(f'  {topic:15s}: {len(p)} pages')
        except Exception as e:
            print(f'  {topic:15s}: ERROR {e}')
            topic_pages[topic] = []

    print('\nPhase 2 — Scraping')
    rows = []
    for topic, pages in topic_pages.items():
        for page in tqdm(pages, desc=f'  {topic}'):
            infobox = scrape(page)
            if not infobox: continue

            classified = []
            for key, value in infobox.items():
                vt, nt = classify(key, value)
                classified.append((key, value, vt, nt))

            # Type-diverse selection
            seen, selected = set(), []
            for item in classified:
                if item[2] not in seen:
                    selected.append(item)
                    seen.add(item[2])
            for item in classified:
                if item not in selected:
                    selected.append(item)
                if len(selected) >= MAX_FACTS_PER_PAGE: break

            for key, value, vt, nt in selected:
                rows.append({'entity':page,'key':key,'value':value,
                             'value_type':vt,'ner_type':nt,'topic':topic})

    kb = pd.DataFrame(rows)

    # Drop thin entities
    counts = kb.groupby('entity').size()
    thin   = counts[counts < MIN_FACTS_FOR_PASSAGE].index
    if len(thin):
        print(f'Dropping {len(thin)} thin entities')
        kb = kb[~kb.entity.isin(thin)].reset_index(drop=True)

    # Phase 3: preload IWN for all unique words in KB
    if IWN_AVAILABLE:
        all_vals  = kb.value.tolist() + kb.key.tolist()
        word_set  = set()
        for phrase in all_vals:
            word_set.update(phrase.split())
            word_set.add(phrase)    # also try the full phrase
        hindi_words = [w for w in word_set if _DEVA_RE.search(w)]
        IWN_CACHE.preload(hindi_words)
        print(f'IWN cache stats: {IWN_CACHE.stats()}')

    print(f'\nKB: {len(kb)} facts | {kb.entity.nunique()} entities')
    print(kb.value_type.value_counts().to_string())
    kb.to_csv(OUT_DIR/'knowledge_base.csv', index=False, encoding='utf-8')
    return kb


KB = build_knowledge_base()

Phase 1 — Page lists
  entertainment  : 40 pages
  sports         : 40 pages
  politics       : 40 pages
  science        : 40 pages
  literature     : 40 pages
  geography      : 38 pages

Phase 2 — Scraping


  entertainment:   0%|          | 0/40 [00:00<?, ?it/s]

  sports:   0%|          | 0/40 [00:00<?, ?it/s]

  politics:   0%|          | 0/40 [00:00<?, ?it/s]

  science:   0%|          | 0/40 [00:00<?, ?it/s]

  literature:   0%|          | 0/40 [00:00<?, ?it/s]

  geography:   0%|          | 0/38 [00:00<?, ?it/s]

Dropping 21 thin entities


Preloading IWN cache:   0%|          | 0/1957 [00:00<?, ?it/s]

  IWN coverage: 619/1957 (31.6%) words found
IWN cache stats: {'queried': 2380, 'found': 1016, 'coverage': 0.427}

KB: 882 facts | 142 entities
value_type
text            268
place           175
date            163
person          142
organization    110
numeric          24


---
# PART D — HiNER-Typed Entity Pool

## Entity Pool

In [ ]:
class EntityPool:
    """
    All entities grouped by (topic, ner_type).
    NER types assigned by HiNER.

    Distractor selection:
    1. Same (topic, ner_type) bucket — domain + type match
    2. Fallback to any bucket with same ner_type
    3. HiNER same_type() verification on distractor
    4. Cross-contamination check via value→entity mapping
    """

    def __init__(self):
        self._pool: Dict[Tuple, Set[str]] = defaultdict(set)
        self._val_to_entities: Dict[str, Set[str]] = defaultdict(set)

    def build(self, kb: pd.DataFrame):
        typed_rows = kb[kb.value_type.isin(['person','place','organization'])]
        for _, row in tqdm(typed_rows.iterrows(), total=len(typed_rows), desc='Entity pool'):
            ner = row.ner_type if row.ner_type != 'OTHER' else row.value_type.upper()
            self._pool[(row.topic, ner)].add(row.value)
            self._val_to_entities[row.value].add(row.entity)
            # Add entity name to pool
            ent_ner = TYPER.type_value(row.entity)
            self._pool[(row.topic, ent_ner)].add(row.entity)
            self._val_to_entities[row.entity].add(row.entity)

    def get_distractor(self, value: str, topic: str,
                        ner_type: str, entity: str) -> Optional[str]:
        # Primary pool
        candidates = list(self._pool.get((topic, ner_type), set()))
        # Fallback: same ner_type, any topic
        if len(candidates) < 3:
            for (t, nt), vals in self._pool.items():
                if nt == ner_type: candidates.extend(list(vals))

        candidates = [c for c in candidates if c != value and c != entity]

        # Cross-contamination filter
        candidates = [c for c in candidates
                      if entity not in self._val_to_entities.get(c, set())]

        if not candidates: return None
        random.shuffle(candidates)

        # HiNER verification
        if HINER_AVAILABLE:
            for c in candidates[:15]:
                if TYPER.same_type(value, c): return c

        return candidates[0] if candidates else None

    def sizes(self):
        return {str(k): len(v) for k, v in self._pool.items() if v}


EPOOL = EntityPool()
EPOOL.build(KB)
print('\nEntity pool (topic, ner_type):')
for k, n in sorted(EPOOL.sizes().items()):
    print(f'  {k:45s}: {n}')

---
# PART E — Template Engine with IWN Key Variation

## Three Template Families

In [ ]:
# Family A — evidence (active voice, IWN key synonyms for surface variation)
T_A = {
    'date':         ['{e} का {k} {v} है।', '{e} के {k} के रूप में {v} दर्ज है।'],
    'numeric':      ['{e} का {k} {v} है।', '{e} के {k} का आँकड़ा {v} है।'],
    'person':       ['{e} के {k} का नाम {v} है।', '{e} का {k} {v} है।'],
    'place':        ['{e} का {k} {v} में स्थित है।', '{e} के {k} के रूप में {v} दर्ज है।'],
    'organization': ['{e} का {k} {v} है।', '{e} से संबद्ध {k} {v} है।'],
    'text':         ['{e} का {k} {v} है।', '{e} के {k} के रूप में {v} जाना जाता है।'],
}

# Family B — claims (passive, reported, nominalized — different from A)
T_B = {
    'date':         ['{v} को {e} का {k} बताया जाता है।',
                     'जानकारी के अनुसार {e} का {k} {v} रहा है।',
                     '{e} के अनुसार, {k} {v} था।'],
    'numeric':      ['{e} की {k} {v} आँकी गई है।',
                     'रिपोर्ट के अनुसार {e} का {k} {v} है।'],
    'person':       ['{v} को {e} का {k} कहा जाता है।',
                     '{e} के {k} के तौर पर {v} का उल्लेख मिलता है।'],
    'place':        ['{e} का {k} {v} को माना जाता है।',
                     '{v} को {e} का {k} कहा जाता है।'],
    'organization': ['{v} को {e} का {k} माना जाता है।',
                     '{e} का {k} {v} बताया जाता है।'],
    'text':         ['{e} को {v} के रूप में {k} में जाना जाता है।',
                     'विशेषज्ञों के अनुसार {e} का {k} {v} रहा है।'],
}

# Family C — paraphrase (relative clause, embedded — most structurally different)
T_C = {
    'date':         ['{e}, जिनका {k} {v} है, एक प्रसिद्ध व्यक्तित्व हैं।',
                     '{v} में {e} का {k} हुआ।'],
    'numeric':      ['{e} की {k} लगभग {v} दर्ज की गई है।',
                     '{k} के मामले में {e} का आँकड़ा {v} बताया गया है।'],
    'person':       ['{e} के {k} के रूप में {v} की भूमिका रही है।',
                     '{v} ही {e} के {k} हैं।'],
    'place':        ['{e} की {k} {v} में स्थित मानी जाती है।',
                     '{v} नगर में {e} का {k} है।'],
    'organization': ['{e} का संबंध {k} के रूप में {v} से है।'],
    'text':         ['{e} की {k} में पहचान {v} की है।',
                     '{e} को {k} के क्षेत्र में {v} माना जाता है।'],
}


def fill(family: dict, vtype: str, e: str, k: str, v: str,
         vary_key: bool = False) -> str:
    """
    Fill a template.

    If vary_key=True (used for evidence Family A):
    Look up an IWN synonym of the key word and use it instead.
    This makes the evidence key surface form different from the claim key.
    Example: claim says 'पत्नी', evidence says 'धर्मपत्नी' or 'जीवनसाथी'.
    """
    templates = family.get(vtype, family.get('text', ['{e} का {k} {v} है।']))
    template  = random.choice(templates)

    if vary_key and IWN_AVAILABLE:
        for kw in k.split():
            if not _DEVA_RE.search(kw): continue
            syn = IWN_CACHE.get_key_synonym(kw)
            if syn:
                k = k.replace(kw, syn, 1)
                break

    return template.format(e=e, k=k, v=v)


# Test
print('Template test — same fact, three families:')
for name, fam, vary in [('A-evidence',T_A,True),('B-claim',T_B,False),('C-paraph',T_C,False)]:
    s = fill(fam,'date','विराट कोहली','जन्म','5 नवंबर 1988', vary_key=vary)
    print(f'  [{name}] {s}')

---
# PART F — Five Transformations

## T1: Numerical Mutation

In [ ]:
def mutate_number(value: str, vtype: str, kb_vals: Set[str]) -> Tuple[Optional[str], str]:
    """
    Mutate a number in the value.

    Bounds by semantic type:
    - date + 4-digit number (year): ±3–15 years
    - date + 2-digit number (day):  ±2–10 days
    - numeric:                      ±10–30%
    """
    num_pat = re.compile(r'[0-9०-९][0-9,०-९\.]*')
    nums = num_pat.findall(value)
    if not nums: return None, 'no_number'

    def priority(n):
        l = n.translate(D2L).replace(',','')
        if len(l) == 4: return 0
        if len(l) <= 2: return 1
        return 2

    target = min(nums, key=priority)
    latin  = target.translate(D2L).replace(',','')
    try: n = int(float(latin))
    except: return None, 'parse_error'
    if n == 0: return None, 'zero'

    is_year = (1000 <= n <= 2100)
    if vtype == 'date' and is_year: delta = random.randint(3, 15)
    elif vtype == 'date':            delta = random.randint(2, 10)
    else:                            delta = max(1, int(abs(n) * random.uniform(0.10, 0.30)))

    mutated_n = n + random.choice([-1, 1]) * delta
    if mutated_n <= 0: mutated_n = n + delta

    is_deva   = any(c in DEVA_DIGITS for c in target)
    mutated_s = str(mutated_n).translate(L2D) if is_deva else str(mutated_n)
    mutated_v = value.replace(target, mutated_s, 1)

    if mutated_v in kb_vals: return None, 'mutated_in_kb'
    return mutated_v, f"'{target}'→'{mutated_s}'"


print('Numerical mutation test:')
for v, vt in [('5 नवंबर 1988','date'),('38,852 वर्ग किमी','numeric')]:
    mv, d = mutate_number(v, vt, set())
    print(f'  {v} → {mv}  [{d}]')

## T2: Entity Swap

In [ ]:
def swap_entity(value: str, ner_type: str, topic: str, entity: str) -> Tuple[Optional[str], str]:
    """Replace entity with HiNER-verified same-type domain-aware distractor."""
    d = EPOOL.get_distractor(value, topic, ner_type, entity)
    if d is None: return None, 'no_distractor'
    return d, f"swap '{value}'→'{d}'"


print('Entity swap test:')
for val, ner, topic, entity in [
    ('जया बच्चन','PERSON','entertainment','अमिताभ बच्चन'),
    ('तिरुवनंतपुरम','LOCATION','geography','केरल'),
]:
    d, det = swap_entity(val, ner, topic, entity)
    print(f'  {val:25s} → {d}  [{det}]')

## T3: Relation Mutation (IWN-driven)

In [ ]:
def mutate_relation(entity: str, key: str, value: str,
                     kb: pd.DataFrame) -> Tuple[Optional[str], Optional[str], str]:
    """
    Swap the relationship key while keeping entity and value constant.

    Candidate replacement keys come from:
    1. IWN siblings of the key headword — co-hyponyms in IWN
       Example: siblings of 'पत्नी' = माता, बहन, पुत्री (all family members)
       → Replace key 'पत्नी' with 'माता', keep same person name as value
       → Now claim says 'X की माता जया बच्चन है' (wrong relationship)

    2. IWN synonyms of the key — different surface form, same meaning
       (only used if the synonym is NOT a strict synonym to ensure semantic shift)

    3. All other KB keys that have the same value_type
       (structurally compatible: both expect PERSON, or both expect LOCATION)

    Cross-validation: entity must NOT already have (cand_key → value) in KB.
    """
    key_hw = key.split()[0]  # headword of the key
    iwn_siblings = IWN_CACHE.siblings(key_hw)
    iwn_synonyms = set(IWN_CACHE.synonyms(key_hw))

    # Get value_type of this key in KB
    orig_row = kb[(kb.entity == entity) & (kb.key == key)]
    if orig_row.empty: return None, None, 'not_in_kb'
    orig_vtype = orig_row.iloc[0].value_type

    # All KB keys with same value_type
    same_type_keys = list(kb[kb.value_type == orig_vtype].key.unique())

    # Candidates = IWN siblings + same-type KB keys (not pure synonyms)
    candidates = list(set(iwn_siblings + same_type_keys))
    candidates = [c for c in candidates if c != key
                  and _DEVA_RE.search(c)
                  and c not in iwn_synonyms]  # skip synonyms (same meaning → not wrong)

    if not candidates: return None, None, 'no_candidates'

    # Cross-validation
    entity_kv = {r.key: r.value for _, r in kb[kb.entity == entity].iterrows()}
    random.shuffle(candidates)

    for ck in candidates[:20]:
        if entity_kv.get(ck) == value: continue  # would be accidentally true
        return ck, value, f"relation '{key}'→'{ck}' (value='{value}' unchanged)"

    return None, None, 'all_cross_val_failed'


# Test
print('Relation mutation test (IWN siblings as candidate keys):')
sample = KB[KB.value_type.isin(['person','place'])].sample(4, random_state=42)
for _, row in sample.iterrows():
    mk, mv, det = mutate_relation(row.entity, row.key, row.value, KB)
    print(f'  [{row.entity[:15]}] key={row.key} → {mk} | val={row.value} | {det}')

## T4: Semantic Contrast (IWN co-hyponyms)

In [ ]:
def semantic_contrast(value: str, kb_vals: Set[str]) -> Tuple[Optional[str], str]:
    """
    Find a semantic contrast using IndoWordNet — zero hardcoding.

    For each word in the value:
    1. Query IWN for co-hyponyms (siblings) — same semantic category,
       different specific concept. These are mutually exclusive alternatives.
       Example: 'बल्लेबाज' siblings in IWN = 'गेंदबाज', 'विकेटकीपर'

    2. If no siblings, try IWN 'similar' relation.

    Replace the found word with its IWN contrast.
    Cross-validate: mutated value must not be in KB for this entity.

    Note on antonyms:
    pyiwn's Lemma.antonym() raises NotImplementedError.
    Co-hyponyms (siblings) serve the same purpose:
    they are semantically contrasting alternatives in the same domain.
    """
    for word in value.split():
        if not _DEVA_RE.search(word): continue

        contrast = IWN_CACHE.get_semantic_contrast(word)
        if not contrast or contrast == word: continue
        if not _DEVA_RE.search(contrast): continue

        mutated = value.replace(word, contrast, 1)
        if mutated == value: continue
        if mutated in kb_vals: continue

        return mutated, f"IWN-contrast '{word}'→'{contrast}'"

    return None, 'no_iwn_contrast'


# Test on text-type values
print('Semantic contrast test (IWN co-hyponyms — no hardcoding):')
text_vals = KB[KB.value_type == 'text'].value.sample(10, random_state=1).tolist()
for val in text_vals:
    mv, det = semantic_contrast(val, set())
    if mv: print(f'  {val:30s} → {mv}  [{det}]')

## T5: Paraphrase + Corruption

In [ ]:
def paraphrase_corrupt(entity: str, key: str, value: str,
                        vtype: str, ner: str, topic: str,
                        kb_vals: Set[str]) -> Tuple[Optional[str], Optional[str], str]:
    """Template C + best available corruption."""
    corrupted = detail = None

    if vtype in ('date','numeric'):
        corrupted, detail = mutate_number(value, vtype, kb_vals)
    elif vtype in ('person','place','organization'):
        corrupted, detail = swap_entity(value, ner, topic, entity)
    elif vtype == 'text':
        corrupted, detail = semantic_contrast(value, kb_vals)

    if corrupted is None: return None, None, f'corrupt_failed:{detail}'
    return fill(T_C, vtype, entity, key, corrupted), corrupted, f'paraphrase+{detail}'


print('Paraphrase+corruption test:')
for _, row in KB.sample(4, random_state=5).iterrows():
    kv = set(KB[KB.entity == row.entity].value)
    claim, cv, det = paraphrase_corrupt(row.entity,row.key,row.value,row.value_type,row.ner_type,row.topic,kv)
    print(f'  [{row.value_type}] {row.value} → {cv}')
    if claim: print(f'  Claim: {claim}\n')

---
# PART G — Validator + Evidence DB + Claims DB + Final Assembly

## Five-Point Validator

In [ ]:
def lexical_overlap(a: str, b: str, n: int = 3) -> float:
    def ng(s): return set(s[i:i+n] for i in range(max(0,len(s)-n+1)))
    g1, g2 = ng(a), ng(b)
    if not g1 or not g2: return 0.0
    return len(g1 & g2) / len(g1 | g2)


def validate(true_claim, false_claim, orig_val, mut_val, evidence):
    if mut_val and mut_val in evidence:   return False, 'check1:mut_in_evidence'
    if orig_val and orig_val not in evidence: return False, 'check2:orig_not_in_evidence'
    if false_claim.strip() == true_claim.strip(): return False, 'check3:no_change'
    if lexical_overlap(false_claim, true_claim) > 0.98: return False, 'check4:overlap_too_high'
    if not _DEVA_RE.search(false_claim): return False, 'check5:no_devanagari'
    return True, 'pass'


print('Validator ready')

## Evidence Database

In [ ]:
def build_evidence_db(kb):
    records = []
    for entity, group in tqdm(kb.groupby('entity'), desc='Evidence passages'):
        sentences = []
        for _, row in group.iterrows():
            # Family A + IWN key synonym variation
            sentences.append(fill(T_A, row.value_type, entity, row.key, row.value, vary_key=True))
        random.shuffle(sentences)
        records.append({'entity': entity, 'passage': ' '.join(sentences),
                         'n': len(sentences), 'topic': group.topic.iloc[0]})
    ev = pd.DataFrame(records)
    print(f'Evidence DB: {len(ev)} passages | avg {ev.n.mean():.1f} sentences | avg {ev.passage.str.len().mean():.0f} chars')
    ev.to_csv(OUT_DIR/'evidence_db.csv', index=False, encoding='utf-8')
    return ev


EV_DB     = build_evidence_db(KB)
EV_LOOKUP = dict(zip(EV_DB.entity, EV_DB.passage))

## Claims Database

In [ ]:
COMPAT = {
    'date':         [('numerical_mutation',0.60),('paraphrase_corruption',0.40)],
    'numeric':      [('numerical_mutation',0.60),('paraphrase_corruption',0.40)],
    'person':       [('entity_swap',0.40),('relation_mutation',0.35),('paraphrase_corruption',0.25)],
    'place':        [('entity_swap',0.40),('relation_mutation',0.35),('paraphrase_corruption',0.25)],
    'organization': [('entity_swap',0.50),('relation_mutation',0.30),('paraphrase_corruption',0.20)],
    'text':         [('semantic_contrast',0.60),('paraphrase_corruption',0.40)],
}

def pick_transform(vtype, exclude=None):
    opts = COMPAT.get(vtype,[('paraphrase_corruption',1.0)])
    if exclude: opts = [(t,w) for t,w in opts if t not in exclude] or opts
    names, weights = zip(*opts)
    total = sum(weights)
    return random.choices(names, weights=[w/total for w in weights], k=1)[0]


def gen_false(entity, key, value, vtype, ner, topic, true_claim, evidence, kb, used):
    kb_vals = set(kb[kb.entity==entity].value)

    for _ in range(8):
        t = pick_transform(vtype, exclude=used)
        claim = mv = None; detail = ''
        try:
            if t == 'numerical_mutation':
                mv, detail = mutate_number(value, vtype, kb_vals)
                if mv: claim = fill(T_B, vtype, entity, key, mv)
            elif t == 'entity_swap':
                mv, detail = swap_entity(value, ner, topic, entity)
                if mv: claim = fill(T_B, vtype, entity, key, mv)
            elif t == 'relation_mutation':
                mk, mv, detail = mutate_relation(entity, key, value, kb)
                if mk: claim = fill(T_B, vtype, entity, mk, value)
            elif t == 'semantic_contrast':
                mv, detail = semantic_contrast(value, kb_vals)
                if mv: claim = fill(T_B, vtype, entity, key, mv)
            elif t == 'paraphrase_corruption':
                claim, mv, detail = paraphrase_corrupt(entity,key,value,vtype,ner,topic,kb_vals)
        except: continue

        if not claim or not mv: continue
        ok, reason = validate(true_claim, claim, value, mv, evidence)
        if ok:
            return {'entity':entity,'key':key,'original_value':value,'mutated_value':mv,
                    'value_type':vtype,'ner_type':ner,'topic':topic,
                    'claim':claim,'label':'REFUTES','transformation':t,'detail':detail}
    return None


def build_claims_db(kb):
    rows, stats, rid = [], defaultdict(int), 0
    for entity, group in tqdm(kb.groupby('entity'), desc='Claims'):
        evidence = EV_LOOKUP.get(entity,'')
        if not evidence: continue
        for _, row in group.iterrows():
            true_claim = fill(T_B, row.value_type, entity, row.key, row.value)
            rid += 1
            rows.append({'id':rid,'entity':entity,'key':row.key,
                         'original_value':row.value,'mutated_value':row.value,
                         'value_type':row.value_type,'ner_type':row.ner_type,'topic':row.topic,
                         'claim':true_claim,'label':'SUPPORTS','transformation':'original','detail':''})
            stats['true'] += 1

            used, ng = set(), 0
            for _ in range(FALSE_PER_TRUE * 4):
                if ng >= FALSE_PER_TRUE: break
                rec = gen_false(entity,row.key,row.value,row.value_type,row.ner_type,
                                row.topic,true_claim,evidence,kb,used)
                if rec:
                    rid += 1; rec['id'] = rid
                    rows.append(rec)
                    used.add(rec['transformation'])
                    stats[f"t_{rec['transformation']}"] += 1
                    stats['false'] += 1; ng += 1
                else:
                    stats['failed'] += 1

    df = pd.DataFrame(rows)
    print(f'Claims DB: {len(df)} | SUPPORTS:{stats["true"]} | REFUTES:{stats["false"]} | Failed:{stats["failed"]}')
    print('Transformations:', {k[2:]:v for k,v in stats.items() if k.startswith('t_')})
    df.to_csv(OUT_DIR/'claims_db.csv', index=False, encoding='utf-8')
    return df


CLAIMS_DB = build_claims_db(KB)

## Final Assembly, Validation, Split, Export

In [ ]:
def assemble(claims):
    df = claims.copy()
    df['evidence'] = df.entity.map(EV_LOOKUP)
    df = df[df.evidence.notna()].reset_index(drop=True)

    sup = df[df.label=='SUPPORTS']
    bad = sup[~sup.apply(lambda r: str(r.original_value) in str(r.evidence), axis=1)]
    print(f'Check A (supports_val in evidence): {len(bad)} dropped')
    df = df.drop(bad.index).reset_index(drop=True)

    ref = df[df.label=='REFUTES']
    bad = ref[ref.apply(lambda r: str(r.mutated_value) in str(r.evidence), axis=1)]
    print(f'Check B (refutes_val not in evidence): {len(bad)} dropped')
    df = df.drop(bad.index).reset_index(drop=True)

    before = len(df)
    df = df.drop_duplicates(subset=['entity','key','label','transformation']).reset_index(drop=True)
    print(f'Check C (dedup): removed {before-len(df)}')

    nahi = df.claim.str.contains('नहीं').sum()
    print(f'Check D (नहीं in any claim): {nahi} — should be 0')

    df['ce_overlap'] = df.apply(lambda r: lexical_overlap(str(r.claim),str(r.evidence)), axis=1)
    print(f'Check E (mean claim-evidence overlap): {df.ce_overlap.mean():.3f} (lower=better)')

    df = df.reset_index(drop=True)
    df['id'] = df.index + 1
    print(f'Final: {len(df)} | SUPPORTS:{(df.label=="SUPPORTS").sum()} | REFUTES:{(df.label=="REFUTES").sum()}')
    return df


def split_pages(df):
    trains, vals, tests = [], [], []
    for topic in df.topic.unique():
        tdf = df[df.topic==topic]
        pages = list(tdf.entity.unique()); random.shuffle(pages)
        nt = max(1,int(len(pages)*0.20)); nv = max(1,int(len(pages)*0.10))
        tests.append(tdf[tdf.entity.isin(pages[:nt])])
        vals.append(tdf[tdf.entity.isin(pages[nt:nt+nv])])
        trains.append(tdf[tdf.entity.isin(pages[nt+nv:])])
    train,val,test = (pd.concat(x).reset_index(drop=True) for x in [trains,vals,tests])
    tp,vp,tep = set(train.entity),set(val.entity),set(test.entity)
    assert not (tp&tep) and not (vp&tep) and not (tp&vp), 'LEAKAGE DETECTED'
    for name,s in [('Train',train),('Val',val),('Test',test)]:
        print(f'  {name}: {len(s)} | {s.entity.nunique()} entities')
    print('Zero entity leakage')
    return train,val,test


FINAL = assemble(CLAIMS_DB)
TRAIN, VAL, TEST = split_pages(FINAL)

# Export NLI format
L2I = {'REFUTES':0,'SUPPORTS':1}
def to_nli(df):
    return [{'premise':str(r.evidence),'hypothesis':str(r.claim),'label':L2I[r.label],
             'label_str':r.label,'entity':r.entity,'topic':r.topic,
             'transformation':r.transformation,'value_type':r.value_type,
             'ner_type':r.ner_type,'original_value':r.original_value,'mutated_value':r.mutated_value}
            for _,r in df.iterrows()]

for name,split in [('train',TRAIN),('val',VAL),('test',TEST)]:
    with open(OUT_DIR/f'nli_{name}.json','w',encoding='utf-8') as f:
        json.dump(to_nli(split),f,ensure_ascii=False,indent=2)
    print(f'nli_{name}.json: {len(split)} records')

FINAL.to_csv(OUT_DIR/'dataset_full.csv',index=False,encoding='utf-8')
FINAL.to_json(OUT_DIR/'dataset_full.json',orient='records',indent=2,force_ascii=False)

meta = {
    'version':'4.0','total':len(FINAL),
    'splits':{'train':len(TRAIN),'val':len(VAL),'test':len(TEST)},
    'labels':{'SUPPORTS':int((FINAL.label=='SUPPORTS').sum()),'REFUTES':int((FINAL.label=='REFUTES').sum())},
    'tools':{'entity_typing':'HiNER cfilt/HiNER-original-xlm-roberta-large',
             'lexical_semantics':'IndoWordNet via pyiwn',
             'iwn_relations':['synonyms/lemma_names','HYPERNYMY','HYPONYMY','SIMILAR','co-hyponyms/siblings']},
    'transformations_used':FINAL[FINAL.label=='REFUTES'].transformation.value_counts().to_dict(),
    'value_types':FINAL.value_type.value_counts().to_dict(),
    'ner_types':FINAL.ner_type.value_counts().to_dict(),
    'zero_negation':True,'page_level_split':True,
    'hiner_available':HINER_AVAILABLE,'indowordnet_available':IWN_AVAILABLE,
    'iwn_cache_stats':IWN_CACHE.stats(),
    'hardcoding':'NONE — all entity types from HiNER, all contrasts from IWN',
}
with open(OUT_DIR/'metadata.json','w',encoding='utf-8') as f:
    json.dump(meta,f,ensure_ascii=False,indent=2)
print('metadata.json saved')
print(f'IWN cache: {IWN_CACHE.stats()}')

## Quality Dashboard

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def quality_report(df):
    print('='*60)
    print('SHORTCUT DETECTION REPORT')
    print('='*60)
    sl = df[df.label=='SUPPORTS'].claim.str.len()
    rl = df[df.label=='REFUTES'].claim.str.len()
    diff = abs(sl.mean()-rl.mean())
    print(f'[1] Claim length SUPPORTS:{sl.mean():.0f} REFUTES:{rl.mean():.0f} diff:{diff:.1f}',
          'ERR' if diff>15 else 'OKAY')
    nahi = df.claim.str.contains('नहीं').sum()
    print(f'[2] नहीं in claims: {nahi}', '⚠' if nahi>0 else '✓ ZERO')
    print(f'[3] Claim-evidence overlap: {df.ce_overlap.mean():.3f}', '⚠' if df.ce_overlap.mean()>0.4 else '✓')
    ref = df[df.label=='REFUTES']
    tdist = ref.transformation.value_counts(normalize=True)
    print('[4] Transformations:')
    for t,p in tdist.items():
        print(f'    {t:28s}: {"█"*int(p*30):30s} {p*100:5.1f}%', 'DOMINANT' if p>0.6 else '')
    print('[5] Topic × Transformation:')
    print(pd.crosstab(ref.topic, ref.transformation, normalize='index').round(2).to_string())

    fig, axes = plt.subplots(2,3,figsize=(18,10))
    fig.suptitle('Hindi CV v4 — Quality Dashboard', fontsize=14, fontweight='bold')
    df.label.value_counts().plot(kind='bar',ax=axes[0,0],color=['#2196F3','#F44336'])
    axes[0,0].set_title('Label Distribution')
    ref.transformation.value_counts().plot(kind='barh',ax=axes[0,1],color='#FF9800')
    axes[0,1].set_title('Transformations (REFUTES)')
    for lbl,col in [('SUPPORTS','#2196F3'),('REFUTES','#F44336')]:
        df[df.label==lbl].claim.str.len().plot(kind='hist',bins=30,alpha=0.6,color=col,label=lbl,ax=axes[0,2])
    axes[0,2].set_title('Claim Length by Label'); axes[0,2].legend()
    df.ce_overlap.plot(kind='hist',bins=30,ax=axes[1,0],color='#9C27B0')
    axes[1,0].axvline(0.5,color='red',ls='--'); axes[1,0].set_title('Claim-Evidence Overlap')
    pd.crosstab(df.topic,df.label).plot(kind='bar',ax=axes[1,1],color=['#2196F3','#F44336'])
    axes[1,1].set_title('Label by Topic'); axes[1,1].tick_params(axis='x',rotation=45)
    df.value_type.value_counts().plot(kind='pie',ax=axes[1,2],autopct='%1.1f%%')
    axes[1,2].set_title('Value Type Distribution'); axes[1,2].set_ylabel('')
    plt.tight_layout()
    plt.savefig(OUT_DIR/'quality_dashboard.png',dpi=150)
    plt.show()
    print(f'Saved: {OUT_DIR}/quality_dashboard.png')


quality_report(FINAL)

## Manual Inspection

In [ ]:
def inspect(df, n=2):
    ref = df[df.label=='REFUTES']
    for t in sorted(ref.transformation.unique()):
        sub = ref[ref.transformation==t].sample(min(n,len(ref[ref.transformation==t])),random_state=42)
        print(f'\n{"─"*65}\n{t.upper()}\n{"─"*65}')
        for _,row in sub.iterrows():
            tc = df[(df.entity==row.entity)&(df.key==row.key)&(df.label=='SUPPORTS')]
            print(f'  Entity  : {row.entity}')
            print(f'  Key     : {row.key}')
            print(f'  True val: {row.original_value}  →  Mutated: {row.mutated_value}')
            print(f'  Detail  : {row.detail}')
            print(f'  Evidence: {str(row.evidence)[:200]}...')
            print(f'  TRUE    : {tc.claim.iloc[0] if len(tc) else "N/A"}')
            print(f'  FALSE   : {row.claim}\n')

inspect(FINAL)

---
## Output Files
```
hindi_cv_v4/
├── knowledge_base.csv   ← entity | key | value | value_type | ner_type | topic
├── evidence_db.csv      ← entity | passage (Family A + IWN key synonym variation)
├── claims_db.csv        ← all TRUE and FALSE claims before final validation
├── dataset_full.csv/json← validated, joined, deduplicated
├── nli_train.json       ← {premise, hypothesis, label} for MuRIL
├── nli_val.json
├── nli_test.json
├── metadata.json        ← full provenance, IWN stats, tool versions
└── quality_dashboard.png
```

## What IndoWordNet Gives Us
| IWN Resource | Replaces |
|---|---|
| `lemma_names()` synonyms of key | Hardcoded key synonym dict |
| HYPERNYMY path of key | Hardcoded key-type group dict |
| HYPONYMY → co-hyponyms (siblings) | Hardcoded antonym map |
| SIMILAR relation | Domain contrast table |
| IWN coverage tells us what works | Manual topic-domain pair lists |

## What HiNER Gives Us
| HiNER Resource | Replaces |
|---|---|
| `type_value()` on infobox values | Rule-based keyword matching on key names |
| `type_key()` combined with IWN | Hardcoded key semantic group lists |
| `same_type()` on distractor | Manual entity type verification rules |